# FTTH — Pipeline complet (Zones B & C, hors ZTD)

Notebook unique de préparation, lancement et pilotage. On l'ouvre, on l'exécute de haut en bas.

1. La première cellule de code installe les paquets listés dans `requirements.txt`, à la racine de l'archive décompressée.
2. La cellule de téléchargement récupère les fichiers réels de l'Arcep (immeubles IPE, contours ZAPM, zonage communal), les convertit et vérifie qu'ils sont exploitables.
3. Les briques 0 à 3 calculent ensuite la cinématique, les quadrants et les tests du Quadrant 2.

`DEMO_MODE` est **désactivé**. Les données synthétiques ne servent que si vous le réactivez explicitement.

Périmètre : hors Zones Très Denses. Zones B (AMII/AMEL) et C (RIP). Maille ZAPM, pas de temps trimestriel de 2018_T2 à 2026_T2.


## 1. Installation des dépendances

Rien à préparer à la main. La cellule suivante :

- retrouve la racine du projet (le dossier qui contient `requirements.txt`, que le notebook soit lancé depuis cette racine ou depuis `notebooks/`) ;
- lance `python -m pip install -r requirements.txt` dans l'interpréteur du notebook ;
- affiche chaque ligne de pip au fur et à mesure.

Si un paquet est déjà installé, pip le signale et passe au suivant. En cas d'échec, la cellule s'arrête : le message au-dessus indique le paquet et la raison.


In [ ]:
# 1 — Installation depuis requirements.txt (racine du projet)
import sys, subprocess, time
from pathlib import Path

def trouver_racine() -> Path:
    ici = Path.cwd().resolve()
    for candidat in [ici, *ici.parents]:
        if (candidat / "requirements.txt").is_file() and (candidat / "src" / "ftth").is_dir():
            return candidat
    raise FileNotFoundError(
        "requirements.txt introuvable au-dessus du dossier courant. "
        "Decompressez l'archive et ouvrez le notebook sans deplacer requirements.txt, src/ ni notebooks/."
    )

ROOT = trouver_racine()
print(f"[racine] {ROOT}", flush=True)
print("[racine] fichiers actifs :", flush=True)
for entree in sorted(ROOT.iterdir()):
    if entree.name.startswith("."):
        continue
    genre = "dossier" if entree.is_dir() else "fichier"
    print(f"           {genre:8} {entree.name}", flush=True)
print(f"[python] {sys.executable}", flush=True)
print(f"[version] {sys.version.split()[0]}", flush=True)
req = ROOT / "requirements.txt"
print(f"[fichier] {req} ({req.stat().st_size} octets)", flush=True)
cmd = [sys.executable, "-m", "pip", "install", "-r", str(req)]
print("[commande] " + " ".join(cmd), flush=True)
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
print(f"[pip] termine en {time.time() - t0:.1f} s, code {code}", flush=True)
if code != 0:
    raise RuntimeError("Installation interrompue. Lisez le journal pip ci-dessus, puis relancez cette cellule.")
src = str(ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
print(f"[import] src ajoute au chemin : {src}", flush=True)


## 2. Paramètres de pilotage

Modifiez uniquement le bloc marqué dans la cellule suivante.

| Paramètre | Défaut | Rôle |
|---|---|---|
| `DEMO_MODE` | `False` | `False` : données Arcep réelles. `True` : territoire synthétique, aucun téléchargement |
| `QUARTERS` | `None` | `None` : tous les millésimes Immeuble publiés (2018_T2 → 2026_T2). Une liste restreint, par exemple `["2024_T4", "2026_T2"]` |
| `DEPARTEMENTS` | `None` | `None` : France entière. Une liste restreint, par exemple `["05", "38"]` ou `["2A", "971"]` |
| `FORCE` | `False` | `True` recalcule même si le parquet existe déjà |
| `K_ROBUST_Z` | 2.0 | seuil du Z-score robuste |
| `MIN_EXCESS_QUARTERS` | 4 | retard pur, en trimestres, qui qualifie aussi une anomalie |

Le calendrier inclusif va de t = 1 (2018_T2) à t = 33 (2026_T2). Le cahier des charges arrondissait cette étendue à 32 trimestres ; le notebook aligne `T_MAX` sur le dernier millésime réellement chargé, pour que la censure à droite tombe juste après la dernière observation.


In [ ]:
# 2 — Parametres. DEMO_MODE reste False pour un essai sur les donnees Arcep.
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
for candidat in [ROOT, *ROOT.parents]:
    if (candidat / "requirements.txt").is_file() and (candidat / "src" / "ftth").is_dir():
        ROOT = candidat
        break
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from ftth.config import FTTHConfig, trimestre_index, trimestre_label

# ===== PARAMETRES =====
DEMO_MODE = False
QUARTERS = None          # None = tous les millesimes Arcep publies
DEPARTEMENTS = None      # None = France entiere. Exemple d'essai : ["05"]
FORCE = False
BASE_DIR = str(ROOT / "data")
POST_QUADRANT_MODE = "AUTO_MODEL"
K_ROBUST_Z = 2.0
MIN_EXCESS_QUARTERS = 4
# ======================

config = FTTHConfig(
    BASE_DIR=Path(BASE_DIR),
    POST_QUADRANT_MODE=POST_QUADRANT_MODE,
    K_ROBUST_Z=K_ROBUST_Z,
    MIN_EXCESS_QUARTERS=MIN_EXCESS_QUARTERS,
    T_MAX=trimestre_index(2026, 2),
)
print(f"[config] racine data     : {config.BASE_DIR.resolve()}", flush=True)
print(f"[config] DEMO_MODE        : {DEMO_MODE}", flush=True)
print(f"[config] QUARTERS         : {QUARTERS if QUARTERS else 'TOUS les millesimes publies'}", flush=True)
print(f"[config] DEPARTEMENTS     : {DEPARTEMENTS if DEPARTEMENTS else 'France entiere'}", flush=True)
print(f"[config] FORCE            : {FORCE}", flush=True)
print(f"[config] calendrier       : t=1 {trimestre_label(1)} → t={config.T_MAX} {trimestre_label(config.T_MAX)}", flush=True)
print(f"[config] zones retenues   : {config.RETAINED_ZONES}", flush=True)
print(f"[config] seuils           : Z >= {config.K_ROBUST_Z} ou retard >= {config.MIN_EXCESS_QUARTERS} trimestres", flush=True)
print(f"[config] mode post        : {config.POST_QUADRANT_MODE}", flush=True)
(config.BASE_DIR / "raw_data").mkdir(parents=True, exist_ok=True)
(config.BASE_DIR / "processed_parquet").mkdir(parents=True, exist_ok=True)
(config.BASE_DIR / "outputs_excel").mkdir(parents=True, exist_ok=True)


In [ ]:
# 2b — Versions effectivement chargees
import duckdb
print(f"[version] duckdb {duckdb.__version__}", flush=True)
from ftth.io_utils import get_duckdb_connection
con = get_duckdb_connection(memory=config.DUCKDB_MEMORY, threads=config.DUCKDB_THREADS, load_spatial=True)
try:
    con.execute("SELECT ST_Point(2.35, 48.86);").fetchone()
    print("[version] extension spatial : chargee (centroides ZAPM et rattachement communal 2018)", flush=True)
except Exception as exc:
    print(f"[version] extension spatial indisponible : {exc}", flush=True)
    if not DEMO_MODE:
        raise RuntimeError("L'extension spatial de DuckDB est necessaire pour les donnees reelles.")
for nom in ("polars", "pyarrow", "pandas", "pyfixest", "py7zr", "openpyxl"):
    try:
        mod = __import__(nom)
        print(f"[version] {nom} {getattr(mod, '__version__', 'ok')}", flush=True)
    except Exception as exc:
        print(f"[version] {nom} absent : {exc}", flush=True)
con.close()


## 3. Téléchargement des données Arcep et contrôle

Source : le jeu data.gouv « Le marché du haut et très haut débit fixe (déploiements) », pas le fichier `base_imb` de Ma connexion internet. Ce dernier liste des adresses, sans état de déploiement ni point de mutualisation. Les fichiers utilisés ici viennent des remontées IPE / ZAPM :

- **Immeuble**, un fichier par trimestre : `imb_etat`, `pm_ref`, `code_l331`, coordonnées. Traduction vers le modèle : `cible` → études, `signe` → programmé, `en cours de deploiement` → en cours, `raccordable demande` et `rad en cours de deploiement` → RAD, `deploye` → raccordable. Les lignes `abandonne` sont écartées.
- **ZAPM** (dernier millésime) : polygones, `RefPM`, opérateur, commune du PM. La distance au PM est la distance euclidienne en Lambert-93 jusqu'au centroïde.
- **Commune** (dernier millésime) : `ZTD` exclu, `L. 33-13` → zone B AMII, code AMEL → zone B AMEL, `ZIPU` → zone C RIP.
- **Référentiel des opérateurs** : le code L. 33-1 (`FRTE`, `SIEA`, …) est remplacé par le nom.

Les millésimes 2018_T2, 2018_T3, 2018_T4, 2019 et 2021_T2 sont des archives 7z et n'ont pas de code INSEE : la commune est alors retrouvée par les coordonnées. Les millésimes plus récents sont des zip et portent `code_insee`.

`QUARTERS = None` télécharge **tous** les millésimes publiés. Comptez de l'ordre de 15 à 25 Go compressés et un long premier passage. Chaque archive déjà convertie en parquet est sautée. Le CSV décompressé n'est pas gardé.

La cellule de vérification s'arrête si un fichier est vide, si un état n'est pas reconnu, si aucune commune B/C n'est chargée, ou si moins de la moitié des PM des immeubles se retrouvent dans les contours ZAPM.


In [ ]:
# 3 — Telechargement, conversion, puis controle.
# Correctif : plusieurs millesimes Arcep (2018_T2 a 2019_T4, 2021_T2) sont des
# archives 7z enregistrees ici sous un nom .zip. Windows n'a pas non plus de
# tube nomme : les vrais zip sont extraits avec la bibliotheque standard.
import os
import shutil
import zipfile
from pathlib import Path

def _format_archive(path):
    entete = Path(path).read_bytes()[:8]
    if entete.startswith(b"PK"):
        return "zip"
    if entete.startswith(b"7z\xbc\xaf'\x1c"):
        return "7z"
    raise RuntimeError(
        f"{path} n'est ni un zip ni un 7z. Supprimez ce fichier dans "
        "data/raw_data/_downloads puis relancez cette cellule."
    )

if DEMO_MODE:
    print("[demo] DEMO_MODE = True : aucun appel au site de l'Arcep.", flush=True)
    print("[demo] Generation d'un territoire synthetique pour inspecter les formules.", flush=True)
    from ftth.synthetic_generator import generate_demo_dataset
    generate_demo_dataset(
        config=config, base_dir=BASE_DIR,
        n_zapm=DEMO_N_ZAPM if "DEMO_N_ZAPM" in dir() else 40,
        seed=config.RANDOM_STATE,
    )
    print("[demo] Fichiers synthetiques ecrits sous data/raw_data. Passez a la Brique 0.", flush=True)
else:
    import ftth.data_acquisition as acquisition
    from ftth.data_acquisition import prepare_real_arcep, verify_real_inputs

    try:
        import py7zr  # noqa: F401
    except ImportError:
        import subprocess, sys
        print("[arcep] installation de py7zr, necessaire pour les archives 7z", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "py7zr"])

    _conversion_origine = acquisition._convert_immeuble_zip

    def _tube_inutile(chemin):
        Path(chemin).parent.mkdir(parents=True, exist_ok=True)

    def _extraire_zip(archive, membre, destination):
        acquisition.step(f"extraction de {membre} vers un fichier temporaire")
        with zipfile.ZipFile(archive) as fichier, fichier.open(membre) as source, open(destination, "wb") as sortie:
            shutil.copyfileobj(source, sortie, length=1024 * 1024)
        return {"thread": None, "proc": None}

    def _convertir(archive, *args, **kwargs):
        genre = _format_archive(archive)
        if genre == "7z" and not str(archive).lower().endswith(".7z"):
            destination = Path(archive).with_suffix(".7z")
            if not destination.exists():
                try:
                    os.link(archive, destination)
                except OSError:
                    shutil.copy2(archive, destination)
            acquisition.step(f"archive 7z enregistree en .zip, lue comme {destination.name}")
            archive = destination
        elif genre != "zip":
            raise RuntimeError(f"{archive} : format {genre} non supporte")
        return _conversion_origine(archive, *args, **kwargs)

    def _rang_csv(nom, taille=0):
        base = str(nom).lower().replace("\\", "/").rsplit("/", 1)[-1]
        if not base.endswith(".csv"):
            return (-1, 0)
        if any(mot in base for mot in ("operateur", "opérateur", "commune", "zapm")):
            return (0, taille)
        priorite = 3 if any(mot in base for mot in ("immeuble", "immeule", "carte_fibre", "mv_imm")) else 1
        return (priorite, taille)

    def _csv_immeubles(archive_zip):
        with zipfile.ZipFile(archive_zip) as fichier:
            infos = [i for i in fichier.infolist() if i.filename.lower().endswith(".csv") and not i.filename.endswith("/")]
        if not infos:
            raise FileNotFoundError(f"Aucun CSV dans {archive_zip}")
        infos.sort(key=lambda i: _rang_csv(i.filename, i.file_size), reverse=True)
        if _rang_csv(infos[0].filename, infos[0].file_size)[0] <= 0:
            raise FileNotFoundError("Pas de CSV d'immeubles : " + ", ".join(i.filename for i in infos))
        if len(infos) > 1:
            acquisition.step(
                "CSV retenu « " + infos[0].filename + " ». Autres fichiers ignores : "
                + ", ".join(i.filename for i in infos[1:])
            )
        return infos[0].filename

    acquisition._zip_csv_member = _csv_immeubles
    acquisition.os.mkfifo = _tube_inutile
    acquisition._stream_zip_csv = _extraire_zip
    acquisition._convert_immeuble_zip = _convertir

    print("[arcep] debut du telechargement reel", flush=True)
    print(f"[arcep] dossier        : {BASE_DIR}", flush=True)
    print(f"[arcep] millesimes     : {QUARTERS if QUARTERS else 'tous'}", flush=True)
    print(f"[arcep] departements   : {DEPARTEMENTS if DEPARTEMENTS else 'tous'}", flush=True)
    summary = prepare_real_arcep(
        base_dir=BASE_DIR,
        quarters=QUARTERS,
        departements=DEPARTEMENTS,
        force=FORCE,
        memory=config.DUCKDB_MEMORY,
        threads=config.DUCKDB_THREADS,
    )
    config.T_MAX = int(summary["t_max_observe"])
    print(
        f"[arcep] T_MAX aligne sur le dernier millesime charge : "
        f"t={config.T_MAX} ({summary['t_max_label']})",
        flush=True,
    )
    for item in summary["imb"]:
        print(
            f"[arcep] charge  {item['label']}  t={item['trimestre']}  {item['lignes']} immeubles  {item['path']}",
            flush=True,
        )
    rapport = verify_real_inputs(BASE_DIR)
    print("[arcep] synthese du controle :", flush=True)
    for cle, valeur in rapport.items():
        print(f"         {cle} = {valeur}", flush=True)


## Brique 0 — ETL géomatique & temporel (fusion des 3 sources)

**Pourquoi 3 sources ?** L'Arcep seul tronque à gauche (un bâtiment absent = encore à l'état **Néant = 0**, pas inexistant) ; la BAN/BD TOPO seule ignore la fibre (`ref_pm`, PM, états). La Brique 0 projette **tous** les bâtiments physiques dans leur ZAPM dès $t=0$ (état 0), puis pivote les 32 millésimes :

$$t_{études}=\min\{t\mid état\in\{études,programmé,en\_cours,rad,raccordable\}\}$$
$$t_{programme}=\min\{t\mid état\in\{programmé,en\_cours,rad,raccordable\}\}$$
$$t_{encours}=\min\{t\mid état\in\{en\_cours,rad,raccordable\}\}$$
$$t_{raccordable}=\min\{t\mid état=raccordable\}\;(99\text{ si jamais, }is\_right\_censored=1)$$
$$ever\_rad=\max_t\mathbb{I}(état=rad),\;duree\_en\_rad=\sum_t\mathbb{I}(état=rad)$$

**Règles** : filtrage hors ZTD (B/C uniquement) ; ancrage $ref\_pm=LAST(ref\_pm\;ORDER\;BY\;t)$ + flag `has_pm_reconfigured` ; $ref\_pm=COALESCE(arcep, géo)$ ; `code_dept` (97/98 → 3 car.) ; `dist_pm_m`.

Sortie : `processed_parquet/step0_etl/batiments_cycles.parquet` (1 ligne = 1 bâtiment).

In [ ]:
# B0.1 — Execution ETL (checkpoint ZSTD, SKIP si existant sauf FORCE=True)
from ftth.brique0_etl import run_brique0, validate_brique0_output
b0_path = run_brique0(config=config, force=FORCE)
print("Sortie B0 :", b0_path)
print(validate_brique0_output(b0_path))


In [ ]:
# B0.2 — Inspection : schema, jalons, censure, zones, operateurs
import duckdb
con = duckdb.connect(":memory:")
print(con.execute(f"SELECT * FROM read_parquet('{b0_path}') LIMIT 3").df().T)
print(con.execute(f"""SELECT zone_reglementaire, COUNT(*) n, AVG(is_right_censored) part_censuree,
  AVG(ever_rad) part_rad, COUNT(DISTINCT ref_pm) zapm, COUNT(DISTINCT operateur_infra) oi
  FROM read_parquet('{b0_path}') GROUP BY 1 ORDER BY 1""").df())
print(con.execute(f"""SELECT operateur_infra, COUNT(*) n FROM read_parquet('{b0_path}')
  GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").df())
con.close()


In [ ]:
# B0.3 (visuel) — Distribution des jalons + part censures
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
df0 = pd.read_parquet(b0_path, columns=["t_etudes","t_encours","t_raccordable","is_right_censored","nb_locaux"])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df0[df0.t_etudes.notna()].t_etudes.hist(bins=32, ax=ax[0]); ax[0].set_title("t_etudes (entree IPE)"); ax[0].set_xlabel("trimestre t")
df0.t_raccordable.replace(99, 33).hist(bins=33, ax=ax[1]); ax[1].set_title("t_raccordable (33 = censure 99)"); ax[1].set_xlabel("trimestre t")
plt.tight_layout(); plt.savefig("b0_jalons.png", dpi=110); print("b0_jalons.png | part censures =", round(float(df0.is_right_censored.mean()), 4))


## Brique 1 — Moteur Pré-Quadrant (cinématique ZAPM, Z-score robuste)

Pour chaque ZAPM $i$ : $t_{0,i}^{ZAPM}=\min_{j\in i}(t_{études,j})$. Pour chaque bâtiment $j$ :

$$D_{j,i}=\min(t_{racc,j},33)-t_{0,i},\;D^{(0\to1)},D^{(1\to3)},D^{(3\to4)}\text{ (localisation du ralentissement)}$$
$$M_i=Médiane(D),\;MAD_i=Médiane(|D-M_i|),\;RetardPur=D-M_i,\;Z^{robuste}=\frac{D-M_i}{\max(1.4826\cdot MAD_i,\;1.0)}$$

$$\text{ANOMALIE\_VITESSE si }Z\ge 2.0\;OU\;RetardPur\ge 4\text{ trim., sinon NOMINAL}$$

**Plancher à 1.0** : si > 50 % de la ZAPM ouvre au même trimestre ($MAD_i=0$), le dénominateur vaut 1 trimestre. **Anti-masquage** : avec 30 % de bâtiments à 14 trim. contre 70 % à 2 trim., $\mu+2\sigma=16.6$ ne détecte **rien**, alors que $M/MAD$ détecte les 30 % (voir test `test_robust_z_no_masking`).

Taux nationaux : $\tau_{IMB}=\frac{\sum\mathbb{I}(anom)}{N}$, $\tau_{Locaux}=\frac{\sum L_j\mathbb{I}(anom)}{\sum L_j}$ (× individuel $L\le2$ / collectif $L\ge3$).

In [ ]:
# B1.1 — Execution Pre-Quadrant (ZAPM stats + classification + anomalies seules)
from ftth.brique1_prequadrant import run_brique1, compute_national_taux
outs1 = run_brique1(config=config, force=FORCE)
print(outs1)
taux = compute_national_taux(outs1["pre_classified"])
for k, v in taux.items(): print(f"{k:24s} = {v:.4f}" if isinstance(v, float) else f"{k:24s} = {v}")


In [ ]:
# B1.2 — Inspection : une ZAPM type (M, MAD, Z) + cas MAD=0 (plancher)
import duckdb
con = duckdb.connect(":memory:")
print("--- 5 ZAPM les plus volumineuses ---")
print(con.execute(f"SELECT * FROM read_parquet('{outs1['zapm_stats']}') ORDER BY nb_batiments_zapm DESC LIMIT 5").df())
print("--- Part des ZAPM avec MAD=0 (plancher actif) ---")
print(con.execute(f"SELECT AVG(CASE WHEN mad_duree_zapm = 0 THEN 1.0 ELSE 0.0 END) part_mad0, COUNT(*) n FROM read_parquet('{outs1['zapm_stats']}')").df())
print("--- Exemple : distribution des D et Z sur la plus grosse ZAPM ---")
z0 = con.execute(f"SELECT ref_pm FROM read_parquet('{outs1['zapm_stats']}') ORDER BY nb_batiments_zapm DESC LIMIT 1").fetchone()[0]
print(con.execute(f"""SELECT statut_pre_quadrant, COUNT(*) n, ROUND(AVG(duree_totale_q),2) D_moy,
  ROUND(AVG(z_score_robuste),2) Z_moy, ROUND(AVG(retard_pur_q),2) retard_moy
  FROM read_parquet('{outs1['pre_classified']}') WHERE ref_pm = '{z0}' GROUP BY 1""").df())
con.close()


In [ ]:
# B1.3 (visuel) — Histogrammes des durees D et Z robuste + verification anti-masquage
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
df1 = pd.read_parquet(outs1["pre_classified"], columns=["duree_totale_q","z_score_robuste","retard_pur_q","statut_pre_quadrant"])
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
df1.duree_totale_q.clip(upper=25).hist(bins=26, ax=ax[0]); ax[0].set_title("D_j,i (trimestres depuis t0 ZAPM)"); ax[0].axvline(df1.duree_totale_q.median(), color="r", ls="--", label="mediane"); ax[0].legend()
df1.z_score_robuste.clip(-2, 10).hist(bins=48, ax=ax[1]); ax[1].set_title("Z robuste (seuil 2.0)"); ax[1].axvline(2.0, color="r", ls="--")
df1.retard_pur_q.clip(-5, 15).hist(bins=40, ax=ax[1+1]); ax[1+1].set_title("Retard pur (seuil 4 trim.)"); ax[1+1].axvline(4, color="r", ls="--")
plt.tight_layout(); plt.savefig("b1_distributions.png", dpi=110)
print("b1_distributions.png | verifiez : pic modal (cohorte PM) + longue traine, seuil Z=2.0 dans la traine (pas de masquage).")


### Sortie 1A — Export Excel pour audit manuel (Voie 1) & Sortie 1B — Flux Parquet (Voie 2)

- **Voie 1** : `export_anomalies_for_manual_audit(code_dept, operateur)` → classeur `.xlsx` des seuls `ANOMALIE_VITESSE` avec sous-durées $D^{(0\to1)},D^{(1\to3)},D^{(3\to4)}$ (localise le ralentissement : prise en compte, programmation, travaux). Filtrer par département/OI pour rester < 1 048 576 lignes.
- **Voie 2** : `anomalies_vitesse_seules.parquet` (+ `batiments_pre_quadrant.parquet` complet) → entrée Brique 2.

In [ ]:
# B1.4 — Sortie 1A : export audit manuel (exemple : 1er departement, top anomalies)
from ftth.exports_excel import export_anomalies_for_manual_audit, export_zapm_summary_excel
import duckdb
con = duckdb.connect(":memory:")
dept0 = con.execute(f"SELECT code_dept FROM read_parquet('{outs1['pre_classified']}') GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 1").fetchone()[0]
con.close()
audit_xlsx = Path(BASE_DIR) / "outputs_excel" / f"Audit_Manuel_Anomalies_dept{dept0}.xlsx"
export_anomalies_for_manual_audit(outs1["pre_classified"], audit_xlsx, code_dept=dept0, limit=200_000)
zapm_xlsx = Path(BASE_DIR) / "outputs_excel" / "Synthese_ZAPM_PreQuadrant.xlsx"
export_zapm_summary_excel(outs1["zapm_stats"], None, zapm_xlsx)
print("1A OK :", audit_xlsx, "|", zapm_xlsx)
print("1B OK :", outs1["anomalies"], "(sas parquet vers Brique 2)")


## Brique 2 — Moteur Post-Quadrant (Q1/Q2/Q3/Q4)

Croisement avec les blocages déclarés à l'Arcep : $BlocageDéclaré=\mathbb{I}(ever\_rad=1\;\lor\;motif\notin\{AUCUN,NULL\})$.

| Brique 1 | Blocage Arcep | Quadrant | Signification |
|---|---|---|---|
| ANOMALIE | oui | **Q1_Blocage_Exogene_Concordant** | retard réel et justifié (syndic, tiers, voirie/Enedis, RAD légitime) |
| ANOMALIE | non | **Q2_Inefficience_OI_Non_Justifiee** | **cible d'étude** : retard anormal sans justification |
| NOMINAL | oui | Q3_Declaration_Preventive_Faux_RAD | RAD préventif sans retard (validation artificielle de jalons) |
| NOMINAL | non | Q4_Deploiement_Nominal | cohorte standard |

Mode `MANUAL_EXCEL_EXPORT` : pas d'affectation auto (audit humain seul). Mode `AUTO_MODEL` : quadrants + dashboard pour la Brique 3.

In [ ]:
# B2.1 — Execution Post-Quadrant
from ftth.brique2_postquadrant import run_brique2
outs2 = run_brique2(config=config, force=FORCE)
print(outs2)


In [ ]:
# B2.2 — Inspection : ventilation Q1-Q4 (globale, par zone, par OI)
import duckdb
con = duckdb.connect(":memory:")
qp = outs2["quadrants"]
print(con.execute(f"SELECT quadrant_final, COUNT(*) n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) pct FROM read_parquet('{qp}') GROUP BY 1 ORDER BY 1").df())
print(con.execute(f"SELECT zone_reglementaire, quadrant_final, COUNT(*) n FROM read_parquet('{qp}') GROUP BY 1,2 ORDER BY 1,2").df().pivot_table(index="zone_reglementaire", columns="quadrant_final", values="n", aggfunc="sum", fill_value=0))
print(con.execute(f"SELECT operateur_infra, COUNT(*) n, ROUND(100*AVG(CASE WHEN quadrant_final LIKE 'Q2%' THEN 1.0 ELSE 0.0 END),2) pct_Q2 FROM read_parquet('{qp}') GROUP BY 1 ORDER BY 3 DESC").df())
con.close()


In [ ]:
# B2.3 — Dashboard final Excel (synthese OI x zone x quadrant)
from ftth.exports_excel import export_dashboard_final_excel
dash = Path(BASE_DIR) / "outputs_excel" / "Synthese_Finale_Quadrants_OI_Zones.xlsx"
export_dashboard_final_excel(outs2["quadrants"], None, dash)
print("Dashboard :", dash)


## Brique 3 — Économétrie du Quadrant 2 (Tests 1 & 2, SFA)

**Variables.** Bâtiment : `dist_pm_m`, `part_aerien_troncon` (Enedis), `is_individuel = I(L ≤ 2)`, `dist_voisin_m` (isolement/bout de ligne). ZAPM : `m_voirie_par_local` (BD TOPO), `ratio_aerien_zapm`, `pente_moyenne_deg` (BD ALTI), `part_habitat_disperse` (> 50 m du voisin) → `score_difficulte_zapm` (moyenne des Z-scores ou ACP) → `quintile_difficulte` 1–5 ; `rang_portefeuille_oi ∈ [0,1]` (essoufflement budgétaire/industriel).

**Test 1 (intra-ZAPM, $j ∈ Q2 ∪ Q4$, FE `ref_pm`)** : $\mathbb{P}(Q2)=\Lambda(\alpha_i+\beta_1\ln DistPM+\beta_2 Aérien+\beta_3 Individuel+\gamma[\ln DistPM×OI]+\delta[\ln DistPM×Zone])$. $H_0$ : bruit aléatoire ($\beta=0$) ; $H_1$ : écrémage de coût marginal ($\beta_1>0$, plus fort en Zone B). Estimateurs : Logit FE (`feglm`), LPM (`feols`), PPML sur `retard_pur_q` (`fepois`).

**Test 2A (inter-ZAPM)** — spécification complète sur données réelles (~110k ZAPM) ; en démo (n<200) le code bascule automatiquement sur une spécification parcimonieuse (quintiles + OI additifs, FE t0) pour rester identifiable : $\tau_i^{Q2}=\frac{N_i^{Q2}}{N_i^{Q2}+N_i^{Q3}+N_i^{Q4}}$, régression quintiles × OI + score × zone + rang + $\tau^{Q1}$ | dept + $t_0$ (poids $N_{horsQ1}$, cluster dept). $\phi_{k,5}\approx0$ = résilient ; $\psi<0$ = RIP (C) protège mieux qu'AMII (B) à difficulté égale ; $\lambda>0$ = essoufflement.

**Test 2B (SFA)** : $\ln(Durée95_i)=f(inputs)+v_i+u_i$, $u_i\ge0$, $TE_i=e^{-u_i}$. MLE normal/half-normal (Python/`scipy`) + 2e étape OLS de $-\ln TE$ sur OI × difficulté + zone + rang (approximation de $\mu_u(Z)$ ; alternative R/`frontier` documentée).

In [ ]:
# B3.0 — Features ZAPM : score de difficulte, quintiles, rang portefeuille
from ftth.brique3_econometrie import build_zapm_features
zapm_sig = Path(BASE_DIR) / config.RAW_ZAPM_PARQUET_FALLBACK
enedis = Path(BASE_DIR) / config.RAW_ENEDIS_PARQUET
zapm_df = build_zapm_features(outs2["quadrants"], Path(BASE_DIR) / config.OUT_ZAPM_FEATURES,
    zapm_sig if zapm_sig.exists() else None, enedis if enedis.exists() else None,
    method="zscore_mean", n_quintiles=config.N_DIFFICULTY_QUINTILES, duree_quantile=config.SFA_DUREE_QUANTILE)
print(zapm_df[["ref_pm","operateur_infra","zone_reglementaire","nb_bat_hors_Q1","taux_Q2","score_difficulte_zapm","quintile_difficulte","rang_portefeuille_oi"]].head(10))
print(zapm_df.groupby("quintile_difficulte").agg(n=("ref_pm","count"), taux_Q2_moy=("taux_Q2","mean"),
    voirie_loc_med=("m_voirie_par_local","median"), aerien_moy=("ratio_aerien_zapm","mean")))


In [ ]:
# B3.1 — Test 1 intra-ZAPM (Logit FE / LPM / PPML). Echantillon Q2+Q4.
from ftth.brique3_econometrie import run_test1
t1 = run_test1(outs2["quadrants"], config=config)
print("backend :", t1.backend, "| n =", t1.n_obs, "|", t1.summary)
for name, mod in [("LOGIT", t1.logit), ("LPM", t1.lpm), ("PPML", t1.ppml)]:
    print(f"\n===== Test 1 — {name} =====")
    try:
        print(mod.summary() if hasattr(mod, "summary") else mod)
    except Exception as e:
        print(f"(resume indisponible : {e})")


In [ ]:
# B3.2 — Test 2A resilience inter-ZAPM (quintiles x OI)
from ftth.brique3_econometrie import run_test2a
t2a = run_test2a(zapm_df, config=config)
print("backend :", t2a.backend, "| n_zapm =", t2a.n_zapm, "|", t2a.summary)
try:
    print(t2a.resilience.summary() if hasattr(t2a.resilience, "summary") else t2a.resilience)
except Exception as e:
    print("(resume indisponible :", e, ")")


In [ ]:
# B3.3 (visuel) — Taux Q2 par quintile de difficulte x operateur (resilience)
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
piv = zapm_df.pivot_table(index="quintile_difficulte", columns="operateur_infra", values="taux_Q2", aggfunc="mean")
ax = piv.plot(marker="o", figsize=(10, 4)); ax.set_title("Taux Q2 moyen par quintile de difficulte et OI (plat = resilient)")
ax.set_xlabel("quintile_difficulte (1 facile -> 5 difficile)"); ax.set_ylabel("taux_Q2"); plt.tight_layout()
plt.savefig("b3_resilience.png", dpi=110); print("b3_resilience.png"); print(piv.round(3))


In [ ]:
# B3.4 — Test 2B SFA (frontiere stochastique cout/temps -> TE)
from ftth.brique3_econometrie import run_sfa
sfa = run_sfa(zapm_df, config=config)
print("backend :", sfa.backend, "|", sfa.summary)
print("params :", sfa.params)
if sfa.te is not None:
    import pandas as pd
    te_df = zapm_df[["ref_pm","operateur_infra","zone_reglementaire","quintile_difficulte"]].copy()
    te_df["TE"] = sfa.te.to_numpy()
    print(te_df.groupby("operateur_infra").TE.agg(["mean","median","count"]).round(3).sort_values("mean"))
    print(te_df.groupby(["quintile_difficulte","zone_reglementaire"]).TE.mean().round(3))
    te_df.to_parquet(Path(BASE_DIR) / "processed_parquet/step3_econometrie/te_scores.parquet", compression="zstd", index=False)
    # MAJ dashboard avec resilience par quintile
    from ftth.exports_excel import export_dashboard_final_excel
    export_dashboard_final_excel(outs2["quadrants"], Path(BASE_DIR) / config.OUT_ZAPM_FEATURES,
        Path(BASE_DIR) / "outputs_excel" / "Synthese_Finale_Quadrants_OI_Zones.xlsx")


## Conclusion — limites, reproductibilité, checkpoints

**Ce que le pipeline prouve.** La Brique 1 isole un retard **relatif à la cohorte PM** (effet fixe ZAPM implicite, robuste au masquage) ; la Brique 2 sépare le retard **justifié** (Q1) du retard **inexpliqué** (Q2, cible) ; la Brique 3 teste si Q2 est un bruit aléatoire ou un **écrémage de coût marginal** (Test 1), et quels OI / régimes (B vs C) **décrochent** dans les zones difficiles (Test 2 + SFA).

**Limites.** ZAPM de rue uniquement (ZTD exclue : PM d'immeuble, saut d'états) ; `dist_voisin_m`/`voirie`/`pente` = proxys en démo (brancher BD TOPO/BD ALTI/Enedis en production) ; SFA Python half-normal homogène + 2e étape naïve (alternative R/`frontier` à hétéroscédasticité paramétrique si exigée par le jury) ; RAD vs vrai blocage tributaire de la qualité des motifs OI.

**Reproductibilité.** `RANDOM_STATE=42`, seuils `K=2.0` / `4 trim.`, sentinelle `99` / calcul `33`, checkpoints ZSTD ci-dessous. Tests : `pytest -q`, smoke CLI : `python scripts/run_pipeline.py --demo --n-zapm 12`.

In [ ]:
# Checkpoints finaux (reprise-sur-panne) + rappel des sorties Excel
from pathlib import Path
for p in [config.OUT_BATIMENTS_CYCLES, config.OUT_ZAPM_STATS, config.OUT_PRE_CLASSIFIED,
          config.OUT_ANOMALIES_ONLY, config.OUT_QUADRANTS, config.OUT_ZAPM_FEATURES]:
    fp = Path(BASE_DIR) / p
    print(("OK  " if fp.exists() else "ABSENT"), f"{fp} ({fp.stat().st_size/1e6:.1f} Mo)" if fp.exists() else "")
print("--- Excel ---")
for x in sorted((Path(BASE_DIR) / "outputs_excel").glob("*.xlsx")):
    print("OK  ", x.name, f"({x.stat().st_size/1e3:.0f} Ko)")
